<a href="https://colab.research.google.com/github/KalekyeRaychelle/Movie-Recommender/blob/5-add-movie-descriptions-to-the-dataset/dataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from google.colab import files

In [3]:
uploaded= files.upload()

Saving links.csv to links (1).csv
Saving movies.csv to movies (1).csv
Saving ratings.csv to ratings (1).csv
Saving README.txt to README (1).txt
Saving tags.csv to tags (1).csv


### MERGING DATA

In [4]:
import pandas as pd
movies= pd.read_csv("movies.csv")
tags=pd.read_csv("tags.csv")
links=pd.read_csv("links.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [6]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [7]:
print("Movies: ", movies.shape)
print("Tags: ", tags.shape)
print("Links: ", links.shape)


Movies:  (9742, 3)
Tags:  (3683, 4)
Links:  (9742, 3)


In [8]:
linksMerged=movies.merge(links[['movieId','tmdbId']], on='movieId', how='left')

In [9]:
linksMerged.head()

,movieId,title,genres,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0
4,5,Father of the Bride Part II (1995),Comedy,11862.0


In [10]:
cleanTags=tags.drop_duplicates(subset=['movieId','tag'])
groupedTags=cleanTags.groupby('movieId')['tag'].apply(lambda x: ', '.join(x)).reset_index()
groupedTags.columns=['movieId','tags']

In [11]:
linksMerged=linksMerged.merge(groupedTags,on='movieId',how='left')
linksMerged.head()
print(linksMerged.shape)

(9742, 5)


In [12]:
linksMerged.head()

,movieId,title,genres,tmdbId,tags
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,"pixar, fun"
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,"fantasy, magic board game, Robin Williams, game"
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,"moldy, old"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,NaN
4,5,Father of the Bride Part II (1995),Comedy,11862.0,"pregnancy, remake"


In [13]:
print(movies.columns)

Index(['movieId', 'title', 'genres'], dtype='object')


In [14]:
print("Missing tmdbIds:",linksMerged['tmdbId'].isna().sum())
print("Movies with no tags:", linksMerged['tags'].isna().sum())

Missing tmdbIds: 8
Movies with no tags: 8170


In [15]:
linksMerged=linksMerged.dropna(subset=['tmdbId'])
print("Remaining movies:",linksMerged.shape[0])

Remaining movies: 9734


### Adding movie descriptions

In [16]:
import requests

In [17]:
tmdbAPIKey="e0d5e6eacfa02482784d26ced929f833"

In [18]:
def getMoviedescription(tmdbId):
  url=f"https://api.themoviedb.org/3/movie/{int(tmdbId)}?api_key={tmdbAPIKey}"
  response=requests.get(url)
  data=response.json()
  return data.get('overview', None)

print(getMoviedescription(862))

Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.


In [19]:
from tqdm import tqdm
tqdm.pandas()

In [20]:
linksMerged['description']=linksMerged['tmdbId'].progress_apply(getMoviedescription)


100%|██████████| 9734/9734 [18:20<00:00,  8.84it/s]


InvalidIndexError: (['title'], ['description'])

In [24]:
linksMerged.head()

,movieId,title,genres,tmdbId,tags,description
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,"pixar, fun","Led by Woody, Andy's toys live happily in his ..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,"fantasy, magic board game, Robin Williams, game",When siblings Judy and Peter discover an encha...
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,"moldy, old",A family wedding reignites the ancient feud be...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,NaN,"Cheated on, mistreated and stepped on, the wom..."
4,5,Father of the Bride Part II (1995),Comedy,11862.0,"pregnancy, remake",Just when George Banks has recovered from his ...


In [25]:
print("Missing descriptions:", linksMerged['description'].isna().sum())
print("Empty descriptions:", (linksMerged['description'] == '').sum())

Missing descriptions: 113
Empty descriptions: 1


In [26]:
linksMerged=linksMerged.dropna(subset=['description'])
linksMerged=linksMerged[linksMerged['description'] != '']
print("Remaining movies:", linksMerged.shape[0])

Remaining movies: 9620


In [27]:
linksMerged.to_csv('cleanedMovies.csv',index=False)

In [28]:
from google.colab import files
files.download('cleanedMovies.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>